# RQ2: when and why CPT helps

On which held-out credit tables the reference CPT (E1) changes the five-fold score, what those tables share (size, features, class balance or LGD point masses, the original model's score, distance to the training tables), whether the change exceeds seed noise, and which single source tables transfer to which targets (E7). With 17 PD and 8 LGD tables these are descriptive associations, not causal effects.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import pandas as pd
from src.visualize import style, campaign as cp, campaign2 as c2, results2 as r2
from src.visualize.figures import FigureSaver
style.apply()
sink = FigureSaver('rq2/02_when_and_why')
report = cp.NotebookReport('RQ2: when and why CPT helps', sink=sink)

## 1. Table properties

Rows, features, categorical and missing shares, default rate or LGD zero/one mass, and the distance of each held-out table to its partition's training tables in standardised property space.

In [2]:
properties = {track: c2.table_properties(track) for track in ('pd', 'lgd')}
for track, frame in list(properties.items()):
    similarity = c2.corpus_similarity(track, frame)
    report.table(track.upper() + ' table properties', frame)
    report.table(track.upper() + ' distance to the CPT training tables', similarity)
    if not similarity.empty:
        properties[track] = frame.merge(similarity[['test_dataset_id', 'nearest', 'mean']], on='test_dataset_id')
report.add('1. Table properties', 'Computed from the processed credit tables; distances are in standardised property space.')

,test_dataset_id,rows,features,categorical_share,missing_share,default_rate
0,0001.gmsc,150000,10,0.000000,0.022437,0.066840
1,0002.taiwan_creditcard,30000,23,0.000000,0.000000,0.221200
2,0003.vehicle_loan,233154,33,0.090909,0.000996,0.217071
3,0004.lendingclub,9578,13,0.076923,0.000000,0.160054
4,0005.myhom,7000,8,0.125000,0.004375,0.400000
5,0006.hackerearth,532428,31,0.290323,0.077290,0.236327
6,0007.cobranded,80000,47,0.000000,0.188361,0.246213
7,0008.german,1000,20,0.650000,0.000000,0.300000
8,PropPD1,81999,15,0.000000,0.064611,0.276089
9,0010.thomas,1225,14,0.142857,0.000000,0.263673


,test_dataset_id,partition,nearest,mean
0,0010.thomas,0,0.987095,2.793329
1,0013.hmeq,0,1.321282,2.519434
2,0014.algorithmwatch,0,2.037052,3.317136
3,0016.bondora_peer2peer,0,2.419657,3.486403
4,0001.gmsc,1,1.812596,3.105138
5,0002.taiwan_creditcard,1,1.266806,2.357893
6,0003.vehicle_loan,1,1.759241,2.520223
7,0008.german,1,2.974873,4.400158
8,0004.lendingclub,2,1.266806,2.651656
9,0006.hackerearth,2,1.804523,2.983175


,test_dataset_id,rows,features,categorical_share,missing_share,zero_share,one_share
0,PropLGD1,57932,8,0.000000,0.038157,0.217842,0.512687
1,PropLGD2,4637,50,0.460000,0.053090,0.036230,0.037093
2,PropLGD3,2545,2,0.000000,0.000000,0.000000,0.000000
3,PropLGD4,762,64,0.453125,0.110523,0.104987,0.119423
4,PropLGD5,594,64,0.281250,0.002262,0.116162,0.159933
5,0006.lgd_freddie,16002,20,0.400000,0.011727,0.114173,0.080677
6,0007.lgd_lendingclub,5627,17,0.294118,0.003293,0.014750,0.003199
7,0008.SBA_loans_case,686,22,0.181818,0.003313,0.000000,0.058309


,test_dataset_id,partition,nearest,mean
0,PropLGD2,0,1.980617,2.802033
1,PropLGD3,0,2.591518,3.954636
2,PropLGD1,1,4.821560,5.222151
3,0006.lgd_freddie,1,1.759206,2.758397
4,PropLGD5,2,2.082722,3.228442
5,0007.lgd_lendingclub,2,1.592158,2.810403
6,PropLGD4,3,2.272937,3.980620
7,0008.SBA_loans_case,3,1.592158,2.863170


## 2. Gains against table properties (E1)

Question: do tables with more headroom, fewer rows, other class balance or more distance from the training tables gain more?

In [3]:
for track in ('pd', 'lgd'):
    gains = c2.endpoint_gains('reference', track)
    merged = c2.select(gains, arm=0, credit=True, complete=True)
    merged = merged.merge(properties[track], on='test_dataset_id') if not merged.empty and not properties[track].empty else pd.DataFrame()
    fig = r2.gain_vs_properties(merged, track=track, title='Five-fold CPT effect against table properties (E1)')
    cp.show(sink, c2.pages(fig, 'gain_vs_properties',
        'Five-fold change of ' + r2.SCORE[track] + ' from reference CPT (E1, seed 42; CPT minus the original model on '
        'identical rows) per held-out credit table and base model (colour), against the original model\'s score, the '
        'number of rows (log scale), features, categorical share, ' + ('default rate' if track == 'pd' else 'share of '
        'zero losses') + ' and the standardised property distance to the nearest training table of its partition. '
        'Tables with all five paired folds only.'), track=track)
    report.table(track.upper() + ' Spearman correlation of gain with each property', c2.gain_correlations(gains, properties[track]))
report.add('2. Gains against table properties (E1)', 'With 17 PD and 8 LGD tables these are descriptive correlations, not causal effects.')

No measurements available for this section yet.


""


No measurements available for this section yet.


""


## 3. Gains beyond seed noise (E1 and E2)

Per table and model, the gains at seeds 42 (E1) and 43-45 (E2), their range, and whether the effect exceeds that range (all seeds agree in sign and the smallest absolute gain exceeds the range). The per-table figure is in RQ1 results, section 1.

In [4]:
for track in ('pd', 'lgd'):
    report.table(track.upper() + ' gains versus seed noise', c2.seed_noise(track))
report.add('3. Gains beyond seed noise (E1 and E2)', 'Tables only; the per-table figure with seed ranges is drawn in rq1/02_results.')

""


""


## 4. Which sources help which targets (E7)

Question: after 2,500 updates of CPT on one source table (learning rate 1e-6), which other credit tables change? This uses the training monitor (one fixed split, up to 10,000 rows per target), not the five-fold evaluation.

In [5]:
frame = c2.load_trajectories('transfer')
for track in ('pd', 'lgd'):
    long = c2.transfer_long(frame, track)
    fig = r2.transfer_heatmaps(long, track=track, title='Which source tables help which targets (E7)')
    cp.show(sink, c2.pages(fig, 'transfer_matrix',
        'Change of the monitored ' + r2.SCORE[track] + ' of each credit target table (columns) from the start to the '
        'end of 2,500 updates of continued pretraining on one source table (rows; learning rate 1e-6), per base model '
        '(panels). Tables are numbered in the order of the accompanying key, the same number for a table as source and as target. Scored '
        'on one fixed context/query split of up to 10,000 rows per target with two preprocessing views. Blue: '
        'the target improved; red: it deteriorated; grey: not monitored (the source itself). One colour scale for all '
        'panels; not a five-fold evaluation.'), track=track)
    report.table(track.upper() + ' E7 change per source, target and model', long)
    report.table(track.upper() + ' E7 mean change per source and model', long.groupby(['model', 'source', 'scope']).change.agg(['mean', 'median', 'size']).reset_index() if not long.empty else long)
report.add('4. Which sources help which targets (E7)', c2.evidence_note('E7 transfer monitor', frame))

No measurements available for this section yet.


""


""


No measurements available for this section yet.


""


""


## 5. Do similar sources transfer better? (E7 similarity test)

Per model, the Spearman correlation between the change of a target and the property distance between source and target (the space of section 1), over all source-target pairs. Pairs sharing a table are not independent, so the p-value permutes the table labels of the distance matrix (Mantel test). Monitor changes, not five-fold results.

In [6]:
for track in ('pd', 'lgd'):
    pairs = c2.transfer_pairs(c2.transfer_long(frame, track), c2.table_properties(track))
    tests = c2.transfer_similarity(pairs)
    fig = r2.transfer_similarity(pairs, tests, track=track, title='Transfer against source-target distance (E7)')
    cp.show(sink, c2.pages(fig, 'transfer_similarity',
        'Change of the monitored ' + r2.SCORE[track] + ' of each credit target after 2,500 updates of continued '
        'pretraining on one source table (learning rate 1e-6), against the distance between source and target in '
        'standardised property space (log rows, log features, categorical and missing shares, target summary), per '
        'base model (panels); one point per source-target pair. ρ: Spearman correlation; p: two-sided Mantel '
        'permutation test over table labels. One fixed split per target, not a five-fold evaluation.'), track=track)
    report.table(track.upper() + ' E7 similarity test per model', tests)
    report.table(track.upper() + ' E7 source-target pairs', pairs)
report.add('5. Similarity test (E7)', c2.evidence_note('E7 similarity test', frame))

No measurements available for this section yet.


""


""


No measurements available for this section yet.


""


""


## Summary

Every displayed table and figure, in section order.

In [7]:
print(report.summary(sink))

RQ2: when and why CPT helps

1. Table properties
Computed from the processed credit tables; distances are in standardised property space.

Table: PD table properties
CSV (first column: row index)
row,test_dataset_id,rows,features,categorical_share,missing_share,default_rate
0,0001.gmsc,150000,10,0,0.022436667,0.06684
1,0002.taiwan_creditcard,30000,23,0,0,0.2212
2,0003.vehicle_loan,233154,33,0.090909091,0.00099570033,0.21707112
3,0004.lendingclub,9578,13,0.076923077,0,0.16005429
4,0005.myhom,7000,8,0.125,0.004375,0.4
5,0006.hackerearth,532428,31,0.29032258,0.077289929,0.23632679
6,0007.cobranded,80000,47,0,0.1883609,0.2462125
7,0008.german,1000,20,0.65,0,0.3
8,PropPD1,81999,15,0,0.064610544,0.27608873
9,0010.thomas,1225,14,0.14285714,0,0.26367347
10,PropPD2,105471,64,0,0.0083016007,0.092755355
11,0012.home_credit,307511,64,0.140625,0.18749543,0.080728819
12,0013.hmeq,5960,12,0.16666667,0.073699664,0.19949664
13,0014.algorithmwatch,158700,64,0,0,0.37809074
14,0015.credit_risk_dataset,325